# **Xây dựng mô hình phân cụm K-means trên tập dữ liệu chim cánh cụt**

In [325]:
import numpy as np 
import pandas as pd 
import plotly.express as px 
import seaborn as sns 
import plotly.graph_objects as go 
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.cluster import KMeans 


In [326]:
data = pd.read_csv("penguins.csv") 
data.head().style.background_gradient(cmap=sns.cubehelix_palette(as_cmap=True)) 

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,39.100000,18.700000,181.000000,3750.000000,MALE
1,39.500000,17.400000,186.000000,3800.000000,FEMALE
2,40.300000,18.000000,195.000000,3250.000000,FEMALE
3,nan,nan,nan,nan,nan
4,36.700000,19.300000,193.000000,3450.000000,FEMALE


## **EDA và xử lý dữ liệu**

In [327]:
#Hiển thị một số thông tin về dữ liệu
# shape
print(f'+ Shape: {data.shape}')
# types
print(f'+ Data Types: \n{data.dtypes}')
# head, tail
print(f'+ Contents: ')
display(data.head(5))
display(data.tail(5))
# info
data.info()

+ Shape: (344, 5)
+ Data Types: 
culmen_length_mm     float64
culmen_depth_mm      float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
dtype: object
+ Contents: 


,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,39.1,18.7,181.0,3750.0,MALE
1,39.5,17.4,186.0,3800.0,FEMALE
2,40.3,18.0,195.0,3250.0,FEMALE
3,NaN,NaN,NaN,NaN,NaN
4,36.7,19.3,193.0,3450.0,FEMALE


,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
339,NaN,NaN,NaN,NaN,NaN
340,46.8,14.3,215.0,4850.0,FEMALE
341,50.4,15.7,222.0,5750.0,MALE
342,45.2,14.8,212.0,5200.0,FEMALE
343,49.9,16.1,213.0,5400.0,MALE


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   culmen_length_mm   342 non-null    float64
 1   culmen_depth_mm    342 non-null    float64
 2   flipper_length_mm  342 non-null    float64
 3   body_mass_g        342 non-null    float64
 4   sex                335 non-null    object 
dtypes: float64(4), object(1)
memory usage: 13.6+ KB


> **Nhận xét**

1. Về cấu trúc và kích thước dữ liệu (Shape & Info)
* Kích thước: Dữ liệu có 344 dòng (entries) và 5 cột (columns). Đây là một tập dữ liệu khá nhỏ, phù hợp để thực hành và học tập, nhưng cần cẩn thận khi xóa dữ liệu để tránh làm mất đi quá nhiều thông tin.
* Các cột dữ liệu:
    * culmen_length_mm, culmen_depth_mm, flipper_length_mm, body_mass_g: 4 cột này có kiểu float64 (số thực). Đây là tín hiệu tốt vì thuật toán K-means yêu cầu dữ liệu đầu vào phải là số để tính khoảng cách.
    * sex: Kiểu object (chuỗi/kí tự). Đây là biến phân loại (categorical), cần phải xử lý (mã hóa thành số) trước khi đưa vào mô hình K-means.
2. Về dữ liệu bị thiếu (Missing Values - NaN)
Quan sát cột Non-Null Count trong Image 4 và dữ liệu hiển thị trong Image 3, ta thấy rõ vấn đề thiếu dữ liệu:

* Các cột số (culmen_..., flipper_..., body_...): Có 342 giá trị không null trên tổng số 344 dòng.
    * => Có 2 dòng bị thiếu hoàn toàn dữ liệu ở các cột này.
    * Bằng chứng: Trong Image 3, dòng index 3 và dòng index 339 chứa toàn giá trị NaN.
* Cột sex: Chỉ có 335 giá trị không null.
    * => Có 9 dòng bị thiếu thông tin về giới tính (344 - 335 = 9).
    * Lưu ý: 9 dòng này bao gồm cả 2 dòng bị thiếu toàn bộ thông tin ở trên.
3. Nhận xét về chất lượng dữ liệu cho K-means
* Dòng rỗng hoàn toàn (All NaN): Dòng số 3 và 339 là những dòng "rác" thực sự vì chúng không chứa thông tin gì cả. Việc xóa chúng là bắt buộc và an toàn tuyệt đối.
* Thiếu thông tin sex: Có những dòng có đủ thông số đo lường (culmen, flipper...) nhưng lại thiếu sex.
    * Nếu bạn định dùng sex làm đặc trưng (feature) cho K-means: Bạn buộc phải xóa các dòng này hoặc điền khuyết (impute) bằng mode (giá trị xuất hiện nhiều nhất).
    * Nếu bạn chỉ dùng các chỉ số đo lường để phân cụm: Bạn có thể bỏ cột sex đi và giữ lại các dòng đó để tận dụng dữ liệu.

In [328]:
#Kiểm tra tính toàn vẹn của dữ liệu
# Kiểm tra giá trị null/nan
data_null = data[data.isnull().any(axis=1)]
if not data_null.empty:
    print('Số lượng dòng chứa giá trị null/nan: ', data_null.shape[0])
    display(data_null)
    print("Số lượng giá trị null/nan theo từng cột:")
    print(data.isnull().sum())
else:
    print('Không có giá trị null/nan nào.')

Số lượng dòng chứa giá trị null/nan:  9


,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
3,NaN,NaN,NaN,NaN,NaN
8,34.1,18.1,193.0,3475.0,NaN
10,37.8,17.1,186.0,3300.0,NaN
11,37.8,17.3,180.0,3700.0,NaN
47,37.5,18.9,179.0,2975.0,NaN
246,44.5,14.3,216.0,4100.0,NaN
286,46.2,14.4,214.0,4650.0,NaN
324,47.3,13.8,216.0,4725.0,NaN
339,NaN,NaN,NaN,NaN,NaN


Số lượng giá trị null/nan theo từng cột:
culmen_length_mm     2
culmen_depth_mm      2
flipper_length_mm    2
body_mass_g          2
sex                  9
dtype: int64


> **Nhận xét:**

Nhìn vào bảng kết quả, ta thấy rõ ràng có 2 kiểu dòng bị lỗi NaN:

* Nhóm 1: Dữ liệu "rác" hoàn toàn (Index 3 và 339)

    * Các dòng này nhận giá trị NaN ở tất cả các cột.
    * Đánh giá: Đây là những dòng vô giá trị. Chúng không cung cấp bất kỳ thông tin nào cho mô hình.
    * Hành động: Bắt buộc phải xóa.
* Nhóm 2: Dữ liệu khuyết thiếu thuộc tính (Index 8, 10, 11, 47, 246, 286, 324)

    * Các dòng này có đầy đủ thông số đo lường (culmen, flipper, mass) nhưng lại thiếu thông tin về giới tính (sex).
    * Đánh giá: Đây là sự mất mát đáng tiếc. Nếu bạn định dùng cột sex làm đặc trưng (feature) để phân cụm, những dòng này trở nên không hợp lệ vì K-means không chấp nhận ô trống.

In [329]:
fig = px.pie(data, 'sex',color_discrete_sequence=['#491D8B','#7D3AC1','#EB548C'],title=
 'Data Distribution',template='plotly') 
fig.show()

> **Nhận xét:**

* MALE (49.1%) và FEMALE (48%):
    * Hai phần này chiếm gần như toàn bộ biểu đồ.
    * Nhận xét: Tỷ lệ giữa chim trống và chim mái rất cân bằng (xấp xỉ 50/50). Đây là một tín hiệu rất tốt cho tập dữ liệu. Sự cân bằng này giúp mô hình học máy (machine learning) không bị thiên lệch (bias) về một giới tính nào, đảm bảo tính đại diện của mẫu dữ liệu.

* Phần màu hồng (null - 2.62%):

    * Đây chính là 9 dòng dữ liệu bị thiếu mà bạn đã tìm ra ở bước trước.
    * Con số 2.62% tương ứng với phép tính: 9÷344≈2.6.
    * Vì tỷ lệ này rất nhỏ (dưới 5%), việc xóa bỏ chúng hoàn toàn không ảnh hưởng đáng kể đến tổng thể dữ liệu.

* Phần nhỏ xíu màu tím nhạt (. - 0.291%):

    * Đây chính là 1 dòng dữ liệu chứa ký tự lạ ".".
    * Con số 0.291% tương ứng với phép tính: 1÷344≈0.29.
    * Đây là nhiễu (noise) do lỗi nhập liệu, bắt buộc phải loại bỏ để cột sex chỉ còn lại 2 giá trị chuẩn.

In [330]:
data_dup = data[data.duplicated(keep=False)]
if not data_dup.empty:
    print('Số hàng trùng nhau: ', data.duplicated(keep="first").sum())

    # Nhóm theo toàn bộ các cột
    grouped = data_dup.groupby(list(data.columns))

    # Duyệt và hiển thị từng nhóm
    for i, (key, group) in enumerate(grouped, start=1):
        print(f'\n nhóm trùng lặp: {i} (số dòng: {len(group)})')
        display(group)

    print('Tổng hợp các dòng trùng lặp')
    display(data_dup)
else:
    print('Không có hàng nào trùng nhau.')

Số hàng trùng nhau:  1
Tổng hợp các dòng trùng lặp


,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
3,NaN,NaN,NaN,NaN,NaN
339,NaN,NaN,NaN,NaN,NaN


In [331]:
# Các tính chất thống kê trên dữ liệu số
description = data.describe().T
display(description)

,count,mean,std,min,25%,50%,75%,max
culmen_length_mm,342.0,43.921930,5.459584,32.1,39.225,44.45,48.50,59.6
culmen_depth_mm,342.0,17.151170,1.974793,13.1,15.600,17.30,18.70,21.5
flipper_length_mm,342.0,214.014620,260.558057,-132.0,190.000,197.00,213.75,5000.0
body_mass_g,342.0,4201.754386,801.954536,2700.0,3550.000,4050.00,4750.00,6300.0


> **Nhận xét:**

Phát hiện bất thường (Outliers) ở cột flipper_length_mm

Tại cột thứ 3: flipper_length_mm (chiều dài cánh), sẽ thấy 2 con số vô lý:

* min = -132.0: Chiều dài cánh không thể là số âm. Đây chắc chắn là lỗi nhập liệu.
* max = 5000.0: Đơn vị ở đây là mm. 5000mm tương đương 5 mét! Không có con chim cánh cụt nào có cánh dài 5 mét cả (thông thường chỉ khoảng 180-230mm).

Các cột khác (culmen_..., body_mass_g): Các chỉ số min, max, mean đều nằm trong khoảng hợp lý về mặt sinh học.

In [332]:
# Tìm những dòng có chiều dài cánh > 4000 hoặc < 0
outliers = data[(data['flipper_length_mm'] > 4000) | (data['flipper_length_mm'] < 0)]
print("Các dòng dữ liệu bất thường:")
print(outliers)

Các dòng dữ liệu bất thường:
    culmen_length_mm  culmen_depth_mm  flipper_length_mm  body_mass_g   sex
9               42.0             20.2             5000.0       4250.0  MALE
14              34.6             21.1             -132.0       4400.0  MALE


Đã tìm thấy 2 dòng dữ liệu nhiễu tại dòng 9 và dòng 14

In [333]:
#Các tính chất thống kê trên dữ liệu phân loại
data.describe(include=['O'])

,sex
count,335
unique,3
top,MALE
freq,169


In [334]:
# Tần số xuất hiện của các lớp trong biến mục tiêu
data["sex"].value_counts()

sex
MALE      169
FEMALE    165
.           1
Name: count, dtype: int64

In [335]:
fig = px.box(data_frame=data, x='sex',y='culmen_length_mm', 
color='sex', color_discrete_sequence=['#29066B','#7D3AC1','#EB548C'], orientation='v') 
fig.show() 

> **Nhận xét:**

* Vị trí trung vị (Đường gạch ngang giữa hộp):
    * MALE: Khoảng 46-47mm.
    * FEMALE: Khoảng 43mm.
    * => Nhận xét: Chim trống (Male) có xu hướng mỏ dài hơn chim mái (Female).
* Độ phân tán (Chiều cao của hộp và râu):
    * Cả hai giới tính đều có sự biến thiên khá rộng, nhưng phân phối dữ liệu của MALE nằm ở dải cao hơn so với FEMALE.

In [336]:
fig = px.histogram(data_frame=data, x='culmen_depth_mm', 
           color='sex', 
           color_discrete_sequence=['#491D8B','#7D3AC1','#EB548C'], 
           nbins=50) 
fig.show() 

fig = px.box(data_frame=data, x='sex',y='culmen_depth_mm', 
color='sex', color_discrete_sequence=['#29066B','#7D3AC1','#EB548C'], orientation='v') 
fig.show()

> **Nhận xét:**

Đối với biểu đồ Histogram

* Sự chồng lấn (Overlap):
    * Vùng từ 16mm đến 18mm có sự chồng lấn rất lớn giữa MALE (Tím đậm) và FEMALE (Tím nhạt).
    * Điều này cho thấy: Nếu chỉ dùng riêng lẻ chỉ số "độ sâu mỏ" thì rất khó để phân biệt đực/cái chính xác. Tuy nhiên, K-means sẽ kết hợp nhiều chỉ số (nhiều chiều) nên vấn đề này sẽ được giải quyết.
* Xu hướng tách biệt:
    * FEMALE (Tím nhạt): Có xu hướng tập trung nhiều hơn ở phía bên trái (giá trị nhỏ hơn, khoảng 13-15mm).
    * MALE (Tím đậm): Có xu hướng tập trung nhiều hơn ở phía bên phải (giá trị lớn hơn, khoảng 18-20mm).
    * => Kết luận: Chim trống thường có mỏ sâu (dày) hơn chim mái.

In [337]:
fig = px.box(data_frame=data, x='sex',y='flipper_length_mm', 
color='sex', color_discrete_sequence=['#29066B','#7D3AC1','#EB548C'], orientation='v') 
fig.show()

fig = px.histogram(data_frame=data, x='flipper_length_mm', 
           color='sex', 
           color_discrete_sequence=['#491D8B','#7D3AC1','#EB548C'], 
           nbins=50) 
fig.show() 

> **Nhận xét:**

* Giá trị ngoại lai cực đoan (Extreme Outliers):
    * Tại biểu đồ Box plot: Có thể thấy một điểm dữ liệu ở nhóm MALE vọt lên tới giá trị 5000. Trong thực tế, cánh chim cánh cụt (flipper length) thường chỉ dao động từ 170mm đến 230mm. Giá trị 5000mm (5 mét) là phi lý, có thể do lỗi nhập liệu (thừa số 0 hoặc nhầm đơn vị đo).
    * Cũng tại nhóm MALE, có một điểm dữ liệu nằm dưới mức 0 (khoảng -100 hoặc -200). Chiều dài cánh không thể là số âm.
* Dữ liệu rác/Lỗi nhãn (Dirty Labels):
    * Ở trục hoành (x-axis) của cả hai biểu đồ, ngoài MALE và FEMALE, xuất hiện một nhãn là dấu chấm .. Đây là dữ liệu bị lỗi, có thể do thiếu thông tin giới tính nhưng được điền bằng ký tự đặc biệt thay vì để trống (NaN).

In [338]:
fig = px.box(data_frame=data, x='sex',y='body_mass_g', 
color='sex', color_discrete_sequence=['#29066B','#7D3AC1','#EB548C'], orientation='v') 
fig.show()

fig = px.histogram(data_frame=data, x='body_mass_g', 
           color='sex', 
           color_discrete_sequence=['#491D8B','#7D3AC1','#EB548C'], 
           nbins=50) 
fig.show() 

> **Nhận xét:**

* Không có Outliers cực đoan:
    * Nhìn vào Box plot (Hình 3), các giá trị nằm trong khoảng từ ~2700g đến ~6300g. Đây là các chỉ số sinh học hoàn toàn hợp lý cho chim cánh cụt.
    * Không có giá trị âm, không có giá trị khổng lồ phi lý. Các "râu" (whiskers) của biểu đồ hộp vươn dài nhưng hợp lý, phản ánh sự đa dạng kích thước tự nhiên.

* Sự phân biệt giữa hai giới tính (Sex Dimorphism):
    * Nhóm MALE (Nam) có xu hướng nặng hơn nhóm FEMALE (Nữ). Trung vị (đường kẻ ngang trong hộp) của Nam cao hơn hẳn Nữ.
    * Tuy nhiên, nhìn vào Histogram (Hình 4), bạn sẽ thấy sự chồng lấn (overlap) khá lớn giữa hai nhóm (vùng màu tím pha trộn). Điều này có nghĩa là nếu chỉ dùng mỗi trọng lượng để phân loại giới tính, mô hình sẽ không chính xác tuyệt đối. Nhưng trong K-means (với mục đích phân cụm loài hoặc giới tính), đây là một đặc trưng tốt khi kết hợp với các biến khác.
* Dạng phân phối:
    * Biểu đồ Histogram cho thấy dữ liệu không hoàn toàn tuân theo phân phối chuẩn hình chuông (Bell curve) đơn lẻ mà có vẻ như là sự kết hợp của nhiều đỉnh (multimodal).
    * Lý do: Tập dữ liệu chim cánh cụt (Palmer Penguins) thường gồm 3 loài: Adelie, Chinstrap và Gentoo. Loài Gentoo thường nặng hơn nhiều so với 2 loài còn lại. Sự phân tán rộng này là tín hiệu tốt cho thuật toán phân cụm, vì K-means sẽ dễ dàng tìm ra các nhóm dựa trên sự khác biệt về trọng lượng này.

In [339]:
fig = px.scatter(data_frame=data, x='culmen_length_mm',y='culmen_depth_mm' 
     ,color='sex',size='body_mass_g', template='seaborn', 
     color_discrete_sequence=['#491D8B','#7D3AC1','#EB548C'],) 
fig.update_layout(width=800, height=600, 
                  xaxis=dict(color="#BF40BF"), 
                  yaxis=dict(color="#BF40BF")) 
fig.show()

> **Nhận xét:**

* Sự hình thành các cụm tự nhiên (Natural Clusters):
    * Nếu nhìn kỹ, bạn sẽ thấy dữ liệu dường như chia thành 2 đám mây lớn (hoặc có thể là 3 đám mây nếu kết hợp thêm kiến thức về loài):
        * Đám mây 1 (Góc trên bên trái): culmen_length thấp (35-45mm) nhưng culmen_depth cao (17-21mm). Đây thường là đặc điểm của loài Adelie.
        * Đám mây 2 (Góc dưới ở giữa/phải): culmen_length trung bình đến cao (40-55mm) nhưng culmen_depth thấp (13-16mm). Đây thường là đặc điểm của loài Gentoo.
        * Đám mây 3 (Góc trên bên phải): culmen_length cao (45-55mm) và culmen_depth cũng cao (17-20mm). Đây thường là loài Chinstrap.
* Sự phân tách giới tính (Sex Separation):
    * Trong từng đám mây lớn đó, các điểm màu tím đậm (MALE) có xu hướng nằm lệch về phía trên và bên phải so với màu tím nhạt (FEMALE). Điều này khẳng định con đực thường có mỏ dài và dày hơn con cái.

In [340]:
#Xử lý các dữ liệu null/nan, miss, duplicate
data_copy = data.copy()

n_before = data_copy.shape[0]

data_copy = data_copy.dropna()

# 3. Đếm số dòng sau khi xóa
n_after = data_copy.shape[0]
n_nan = n_before - n_after

print(f"Số dòng ban đầu: {n_before}")
print(f"Số dòng sau khi xóa trùng: {n_after}")
print(f"Số lượng dòng trùng lặp bị loại bỏ: {n_nan}")
print(f"Tỷ lệ mất dữ liệu: {(n_nan/n_before)*100:.2f}%")

Số dòng ban đầu: 344
Số dòng sau khi xóa trùng: 335
Số lượng dòng trùng lặp bị loại bỏ: 9
Tỷ lệ mất dữ liệu: 2.62%


In [341]:
#Xử lý giá trị nhiễu
#Xem dòng chứa ký tự lạ (dấu chấm)
row_strange = data_copy[data_copy['sex'] == '.']
print("Dòng dữ liệu chứa ký tự lạ:")
print(row_strange)

#Xóa dòng đó
#Chỉ giữ lại những dòng mà cột sex KHÁC dấu chấm
data_copy = data_copy[data_copy['sex'] != '.']

# Kiểm tra lại kết quả
print("\nSố lượng các giá trị trong cột sex sau khi xóa:")
print(data_copy['sex'].value_counts())

Dòng dữ liệu chứa ký tự lạ:
     culmen_length_mm  culmen_depth_mm  flipper_length_mm  body_mass_g sex
336              44.5             15.7              217.0       4875.0   .

Số lượng các giá trị trong cột sex sau khi xóa:
sex
MALE      169
FEMALE    165
Name: count, dtype: int64


In [342]:
# Lọc lấy dữ liệu hợp lệ
data_copy = data_copy[(data_copy['flipper_length_mm'] > 0) & (data_copy['flipper_length_mm'] < 1000)]

# Kiểm tra lại thống kê sau khi xóa
print(data_copy.describe())

       culmen_length_mm  culmen_depth_mm  flipper_length_mm  body_mass_g
count        332.000000       332.000000         332.000000   332.000000
mean          44.021084        17.153012         200.975904  4206.475904
std            5.452462         1.960275          14.035971   806.361278
min           32.100000        13.100000         172.000000  2700.000000
25%           39.500000        15.600000         190.000000  3550.000000
50%           44.700000        17.300000         197.000000  4025.000000
75%           48.625000        18.700000         213.000000  4781.250000
max           59.600000        21.500000         231.000000  6300.000000


In [343]:
#Reset Index
data_copy.reset_index(drop=True, inplace=True)

In [344]:
le = LabelEncoder()
data_copy['sex'] = le.fit_transform(data_copy['sex'])

# Kiểm tra kết quả
print(data_copy[['sex']].head())

# Xem nó đã mã hóa cái gì thành số mấy
print("Các lớp:", le.classes_) # ['FEMALE' 'MALE'] => 0 là FEMALE, 1 là MALE

   sex
0    1
1    0
2    0
3    0
4    1
Các lớp: ['FEMALE' 'MALE']


In [345]:
# Bước 2: Khởi tạo Scaler
scaler = StandardScaler()

# Bước 3: Thực hiện chuẩn hóa (Fit & Transform)
X_scaled = scaler.fit_transform(data_copy)

# X_scaled lúc này là một mảng Numpy (numpy array). 
# Để dễ xem, bạn có thể chuyển nó lại thành DataFrame:
df_scaled = pd.DataFrame(X_scaled, columns=data_copy.columns)

# Kiểm tra kết quả
print("Dữ liệu sau khi chuẩn hóa (5 dòng đầu):")
print(df_scaled.head())

# Kiểm tra thống kê (Mean xấp xỉ 0, Std xấp xỉ 1)
print("\nThống kê mô tả:")
print(df_scaled.describe().round(2))

Dữ liệu sau khi chuẩn hóa (5 dòng đầu):
   culmen_length_mm  culmen_depth_mm  flipper_length_mm  body_mass_g       sex
0         -0.903906         0.790360          -1.425342    -0.566948  0.993994
1         -0.830434         0.126187          -1.068577    -0.504847 -1.006042
2         -0.683490         0.432728          -0.426399    -1.187953 -1.006042
3         -1.344738         1.096901          -0.569105    -0.939551 -1.006042
4         -0.867170         1.761074          -0.783164    -0.691149  0.993994

Thống kê mô tả:
       culmen_length_mm  culmen_depth_mm  flipper_length_mm  body_mass_g  \
count            332.00           332.00             332.00       332.00   
mean              -0.00             0.00               0.00         0.00   
std                1.00             1.00               1.00         1.00   
min               -2.19            -2.07              -2.07        -1.87   
25%               -0.83            -0.79              -0.78        -0.82   
50%          

In [346]:
sse = [] 
K_range = range(1, 9)
for i in K_range: 
    kmeans = KMeans(n_clusters=i , max_iter=300) 
    kmeans.fit(df_scaled)  
    sse.append(kmeans.inertia_) 

fig = px.line(x=K_range, y=sse,template="seaborn",title='Elbow Method') 

fig.update_layout(width=800, height=600, 
title_font_color="#BF40BF",  
xaxis=dict(color="#BF40BF", title="Number of Clusters (K)", tickmode='linear'),  
yaxis=dict(color="#BF40BF",title="SSE"))

fig.show()

> **Nhận xét:**

* Giai đoạn giảm mạnh (K=1 đến K=3): Đường cong lao dốc rất nhanh. Điều này cho thấy việc chia dữ liệu thành 2 hoặc 3 nhóm giúp giảm sai số đáng kể. Các nhóm này rất khác biệt nhau.
* Điểm gập (The Elbow): Tại K=3, độ dốc bắt đầu thay đổi rõ rệt.
    * Từ 1 xuống 2: Rất dốc.
    * Từ 2 xuống 3: Vẫn dốc.
    * Từ 3 xuống 4: Độ dốc bắt đầu thoai thoải hơn (ít dốc hơn hẳn so với đoạn trước).
* Giai đoạn bão hòa (K > 4): Sau mức K=4, việc tăng thêm số cụm không làm giảm SSE nhiều nữa (đường biểu đồ bắt đầu đi ngang).

In [347]:
kmeans = KMeans(n_clusters = 3,  
        init = 'k-means++',  
        max_iter = 300, n_init = 10, random_state = 0) 
clusters = kmeans.fit_predict(df_scaled) 



In [348]:

# Thêm nhãn dự đoán vào DataFrame để vẽ (chuyển sang string để Plotly hiểu là danh mục màu)
df_scaled['Cluster'] = kmeans.labels_.astype(str)

# Lấy tọa độ tâm cụm
centroids = kmeans.cluster_centers_

# 1. Vẽ các điểm dữ liệu (Tô màu theo Cluster thay vì Sex)
fig = px.scatter(
    data_frame=df_scaled, 
    x='culmen_length_mm', # Đảm bảo tên cột đúng
    y='culmen_depth_mm',  # Đảm bảo tên cột đúng
    color='Cluster',      # <--- SỬA LẠI CHỖ NÀY (Quan trọng nhất)
    template='seaborn',
    title='K-means Clustering Results (Phân theo Cụm)'
)

# 2. Vẽ thêm các dấu X tâm cụm
fig.add_trace(
    go.Scatter(
        x=centroids[:, 0], # Cột 0 tương ứng với x='culmen_length_mm'
        y=centroids[:, 1], # Cột 1 tương ứng với y='culmen_depth_mm'
        mode='markers',
        marker=dict(color='#CAC9CD', size=15, symbol='x'),
        name='Centroids'
    )
)

fig.update_layout(template='plotly_dark',width=1000, 
height=500,title='Kmean Clustering Results') 

fig.show()

> **Nhận xét:**

* Cụm 2 (Xanh lá): Đặc trưng là Mỏ dài (trục X dương) nhưng Mỏ nông/mỏng (trục Y âm). Đây chắc chắn là loài Gentoo. Loài này thường to lớn và khác biệt hẳn 2 loài kia.
* Cụm 1 (Cam): Đặc trưng là Mỏ ngắn (trục X âm) và Mỏ khá dày (trục Y trung bình/dương). Đây có thể là loài Adelie.
* Cụm 0 (Xanh dương): Đặc trưng là Mỏ dài trung bình đến cao, và Mỏ dày. Đây có thể là loài Chinstrap hoặc những con Adelie to lớn.

In [351]:
import plotly.express as px

# Đừng quên dùng dữ liệu GỐC (chưa scale) để hiển thị tooltip cho dễ hiểu
# Nhưng tô màu (color) thì dùng kết quả label từ K-means

fig = px.scatter_3d(
    df_scaled, # Nên dùng df gốc để hiện số thực (gram, mm)
    x='culmen_length_mm', 
    y='culmen_depth_mm', 
    z='body_mass_g',  # Thêm chiều cân nặng
    color=kmeans.labels_.astype(str), # Tô màu theo cụm K-means tìm được
    color_discrete_sequence=['#491D8B', '#EB548C', '#7D3AC1'],
    title='Kết quả phân cụm K-means trong không gian 3D',
    opacity=0.7
)
fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
fig.show()